In [15]:
import pandas as pd

pd.options.future.infer_string = False

df = pd.read_csv('../data/raw/property_features.csv')
print("Shape:", df.shape)
df.head()

Shape: (20060, 4)


,feature_id,property_id,feature_name,feature_value
0,FTR-00001,PRP-0001,vaulted_ceilings,1
1,FTR-00002,PRP-0001,hardwood_floors,yes
2,FTR-00003,PRP-0002,solar_panels,YES
3,FTR-00004,PRP-0002,master_suite,Yes
4,FTR-00005,PRP-0003,updated_kitchen,yes


In [16]:
# Data type verification
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20060 entries, 0 to 20059
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   feature_id     20060 non-null  object
 1   property_id    20060 non-null  object
 2   feature_name   20060 non-null  object
 3   feature_value  20060 non-null  object
dtypes: object(4)
memory usage: 627.0+ KB


In [17]:
# Null value checks
print("Null counts per column:")
print(df.isnull().sum())
print(f"Total nulls: {df.isnull().sum().sum()}")
print(f"Null % per column:")
print((df.isnull().sum() / len(df) * 100).round(2).astype(str) + "%")

Null counts per column:
feature_id       0
property_id      0
feature_name     0
feature_value    0
dtype: int64
Total nulls: 0
Null % per column:
feature_id       0.0%
property_id      0.0%
feature_name     0.0%
feature_value    0.0%
dtype: object


In [18]:
df.duplicated().sum()

np.int64(0)

In [19]:
# Step 1: Normalize feature_value
print("Before normalization:")
print(df['feature_value'].value_counts())

bool_map = {'yes': '1', 'no': '0', 'true': '1', 'false': '0'}
df['feature_value'] = (
    df['feature_value']
    .str.strip()
    .str.lower()
    .replace(bool_map)
)

print("\nAfter normalization:")
print(df['feature_value'].value_counts())

Before normalization:
feature_value
no     3445
0      3395
yes    3334
YES    3307
1      3301
Yes    3278
Name: count, dtype: int64



After normalization:
feature_value
1    13220
0     6840
Name: count, dtype: int64


In [20]:
# Step 2: Normalize feature_name — strip whitespace and convert to lowercase
# Prevents same feature being treated as different due to casing or extra spaces
print("Unique feature_names before cleaning:", df['feature_name'].nunique())

df['feature_name'] = df['feature_name'].str.strip().str.lower()

print("Unique feature_names after cleaning:", df['feature_name'].nunique())
print("\nAll feature names:")
print(df['feature_name'].value_counts())

Unique feature_names before cleaning: 14
Unique feature_names after cleaning: 14

All feature names:
feature_name
updated_kitchen         1504
master_suite            1475
granite_counters        1472
central_ac              1459
open_floor_plan         1458
fireplace               1451
solar_panels            1438
vaulted_ceilings        1429
new_roof                1421
stainless_appliances    1413
hardwood_floors         1393
fresh_paint             1388
smart_home              1383
walk_in_closet          1376
Name: count, dtype: int64


In [21]:
# Step 3: Detect duplicate (property_id, feature_name) pairs
# Each property should have each feature listed only once
dupes = df.duplicated(subset=['property_id', 'feature_name'], keep=False)
print(f"Duplicate (property_id, feature_name) pairs: {dupes.sum()}")

if dupes.sum() > 0:
    print("\nDuplicate rows:")
    print(df[dupes].sort_values(['property_id', 'feature_name']))
    # Keep the first occurrence and drop the rest
    df = df.drop_duplicates(subset=['property_id', 'feature_name'], keep='first')
    print(f"\nShape after removing duplicates: {df.shape}")

Duplicate (property_id, feature_name) pairs: 0


In [22]:
# Step 4: Validate ID format patterns
# feature_id must follow FTR-XXXXX (5 digits), property_id must follow PRP-XXXX (4 digits)
invalid_ftr = ~df['feature_id'].str.match(r'^FTR-\d{5}$')
invalid_prp = ~df['property_id'].str.match(r'^PRP-\d{4}$')

print(f"Invalid feature_ids: {invalid_ftr.sum()}")
print(f"Invalid property_ids: {invalid_prp.sum()}")

if invalid_ftr.sum() > 0:
    print("\nSample invalid feature_ids:")
    print(df[invalid_ftr]['feature_id'].head(10).tolist())

if invalid_prp.sum() > 0:
    print("\nSample invalid property_ids:")
    print(df[invalid_prp]['property_id'].head(10).tolist())

Invalid feature_ids: 0
Invalid property_ids: 0


In [23]:
# Step 5: Check feature_id uniqueness
# feature_id is the primary key — no two rows should share the same ID
duplicate_ftr_ids = df['feature_id'].duplicated().sum()
print(f"Duplicate feature_ids: {duplicate_ftr_ids}")

if duplicate_ftr_ids > 0:
    print("\nDuplicated feature_id values:")
    print(df[df['feature_id'].duplicated(keep=False)].sort_values('feature_id'))

Duplicate feature_ids: 0


In [24]:
# Step 6: Strip whitespace from all string columns
# Catches any leading/trailing spaces that might have been missed in earlier steps
str_cols = df.select_dtypes(include='object').columns
df[str_cols] = df[str_cols].apply(lambda col: col.str.strip())
print("Whitespace stripped from columns:", str_cols.tolist())
print("\nFinal shape:", df.shape)
print("\nSample cleaned data:")
df.head()

Whitespace stripped from columns: ['feature_id', 'property_id', 'feature_name', 'feature_value']

Final shape: (20060, 4)

Sample cleaned data:


,feature_id,property_id,feature_name,feature_value
0,FTR-00001,PRP-0001,vaulted_ceilings,1
1,FTR-00002,PRP-0001,hardwood_floors,1
2,FTR-00003,PRP-0002,solar_panels,1
3,FTR-00004,PRP-0002,master_suite,1
4,FTR-00005,PRP-0003,updated_kitchen,1


In [25]:
# Data type check — review current dtypes and cast where appropriate
print("Current dtypes:")
print(df.dtypes)
print()

# feature_value contains '0' and '1' strings — cast to integer for proper numeric use
df['feature_value'] = pd.to_numeric(df['feature_value'], errors='coerce')

print("Dtypes after casting feature_value to numeric:")
print(df.dtypes)
print()

# Check if casting introduced any NaNs (would mean unexpected non-numeric values slipped through)
nulls_after_cast = df['feature_value'].isnull().sum()
print(f"NaNs in feature_value after cast: {nulls_after_cast}")
if nulls_after_cast > 0:
    print("Non-numeric values that failed cast:")
    print(df[df['feature_value'].isnull()])

Current dtypes:
feature_id       object
property_id      object
feature_name     object
feature_value    object
dtype: object

Dtypes after casting feature_value to numeric:
feature_id       object
property_id      object
feature_name     object
feature_value     int64
dtype: object

NaNs in feature_value after cast: 0


In [26]:
# Overall dataset assessment — final cleanliness report
print("=" * 50)
print("        DATASET CLEANING ASSESSMENT REPORT")
print("=" * 50)

print(f"\n Shape               : {df.shape[0]} rows x {df.shape[1]} columns")

print(f"\n Null values         :")
print(df.isnull().sum().to_string())

print(f"\n Full-row duplicates : {df.duplicated().sum()}")

prop_feat_dupes = df.duplicated(subset=['property_id', 'feature_name']).sum()
print(f" (property_id, feature_name) duplicates: {prop_feat_dupes}")

print(f"\n feature_id unique   : {df['feature_id'].nunique()} / {len(df)}")

invalid_ftr = ~df['feature_id'].str.match(r'^FTR-\d{5}$')
invalid_prp = ~df['property_id'].str.match(r'^PRP-\d{4}$')
print(f"\n Invalid feature_id format  : {invalid_ftr.sum()}")
print(f" Invalid property_id format : {invalid_prp.sum()}")

print(f"\n feature_name unique values : {df['feature_name'].nunique()}")
print(f" feature_value unique values: {sorted(df['feature_value'].dropna().unique().tolist())}")

print(f"\n Data types:")
print(df.dtypes.to_string())

print("\n" + "=" * 50)
all_clear = (
    df.isnull().sum().sum() == 0 and
    df.duplicated().sum() == 0 and
    prop_feat_dupes == 0 and
    invalid_ftr.sum() == 0 and
    invalid_prp.sum() == 0 and
    df['feature_id'].nunique() == len(df)
)
status = "PASSED — Dataset is clean and ready." if all_clear else "ISSUES FOUND — Review above output."
print(f" Status: {status}")
print("=" * 50)

        DATASET CLEANING ASSESSMENT REPORT

 Shape               : 20060 rows x 4 columns

 Null values         :
feature_id       0
property_id      0
feature_name     0
feature_value    0

 Full-row duplicates : 0
 (property_id, feature_name) duplicates: 0

 feature_id unique   : 20060 / 20060

 Invalid feature_id format  : 0
 Invalid property_id format : 0

 feature_name unique values : 14
 feature_value unique values: [0, 1]

 Data types:
feature_id       object
property_id      object
feature_name     object
feature_value     int64

 Status: PASSED — Dataset is clean and ready.


In [27]:
# Save the cleaned dataframe to the processed folder
df.to_csv('../data/processed/property_features_cleaned.csv', index=False)
print("Cleaned file saved successfully.")
print("Final shape:", df.shape)

Cleaned file saved successfully.
Final shape: (20060, 4)
